# Apple vs Samsung — Competitive Financial & Market Analysis

## Data Collection

**Data Sources:**
1. **Yahoo Finance (via `yfinance`)** — historical stock prices, volume, and financial statements for Apple (AAPL) and Samsung (005930.KS)
2. **FRED (Federal Reserve Economic Data, via `pandas-datareader`)** — macroeconomic context: US Consumer Price Index (CPI) and US 10-Year Treasury Rate

**Coverage:** 2015-01-01 to present

**Variable Definitions:**
- `Open`, `High`, `Low`, `Close`, `Volume` — daily OHLCV stock price data
- `Total Revenue`, `Net Income`, `Gross Profit`, `Operating Income` — annual income statement items
- `Total Assets`, `Total Liab`, `Total Stockholder Equity` — annual balance sheet items
- `Free Cash Flow`, `Operating Cash Flow` — annual cash flow items
- `CPIAUCSL` — US CPI, all urban consumers (monthly, seasonally adjusted)
- `DGS10` — US 10-Year Treasury Constant Maturity Rate (daily)

In [2]:
%pip install yfinance pandas numpy matplotlib seaborn pandas-datareader -q
print('All packages ready.')

Note: you may need to restart the kernel to use updated packages.
All packages ready.


In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
from datetime import datetime

START_DATE = '2015-01-01'
END_DATE   = datetime.today().strftime('%Y-%m-%d')

TICKERS = {
    'Apple':   'AAPL',
    'Samsung': '005930.KS'
}

print(f'Date range: {START_DATE} → {END_DATE}')

Date range: 2015-01-01 → 2026-04-20


## Source 1 — Yahoo Finance: Historical Stock Prices

In [4]:
price_data = {}

for company, ticker in TICKERS.items():
    print(f'Fetching {company} ({ticker})...')
    df = yf.download(ticker, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
    if df.empty:
        print(f'  WARNING: No data returned for {ticker}')
    else:
        # Flatten multi-level columns if present
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
        df.index = pd.to_datetime(df.index)
        price_data[company] = df
        print(f'  {len(df)} rows | {df.index.min().date()} to {df.index.max().date()}')

prices_combined = pd.concat(price_data, axis=1)
prices_combined.to_csv('stock_prices.csv')
print('\nSaved: stock_prices.csv')
prices_combined.tail(3)

Fetching Apple (AAPL)...
  2839 rows | 2015-01-02 to 2026-04-17
Fetching Samsung (005930.KS)...
  2768 rows | 2015-01-02 to 2026-04-17

Saved: stock_prices.csv


Apple                                                  \
                 Close        High         Low        Open      Volume   
Date                                                                     
2026-04-15  266.429993  266.559998  257.809998  258.160004  49913500.0   
2026-04-16  263.399994  267.160004  261.269989  266.799988  43323100.0   
2026-04-17  270.230011  272.299988  266.720001  266.959991  61436200.0   

             Samsung                                            
               Close      High       Low      Open      Volume  
Date                                                            
2026-04-15  211000.0  215500.0  210000.0  215000.0  24092884.0  
2026-04-16  217500.0  218000.0  210500.0  212000.0  21499788.0  
2026-04-17  216000.0  218000.0  215000.0  217000.0  18079322.0

## Source 1 — Yahoo Finance: Financial Statements (Annual)

In [5]:
financials = {}

for company, ticker in TICKERS.items():
    print(f'Fetching financials for {company} ({ticker})...')
    t = yf.Ticker(ticker)

    income_stmt   = t.financials
    balance_sheet = t.balance_sheet
    cash_flow     = t.cashflow

    financials[company] = {
        'income_stmt':   income_stmt,
        'balance_sheet': balance_sheet,
        'cash_flow':     cash_flow
    }

    for label, df in [('income_stmt', income_stmt), ('balance_sheet', balance_sheet), ('cash_flow', cash_flow)]:
        if df is not None and not df.empty:
            fname = f'{company.lower()}_{label}.csv'
            df.to_csv(fname)
            print(f'  Saved {fname}  shape={df.shape}')
        else:
            print(f'  WARNING: {company} {label} is empty')

Fetching financials for Apple (AAPL)...
  Saved apple_income_stmt.csv  shape=(39, 5)
  Saved apple_balance_sheet.csv  shape=(69, 5)
  Saved apple_cash_flow.csv  shape=(53, 5)
Fetching financials for Samsung (005930.KS)...
  Saved samsung_income_stmt.csv  shape=(47, 4)
  Saved samsung_balance_sheet.csv  shape=(82, 4)
  Saved samsung_cash_flow.csv  shape=(59, 5)


## Source 2 — FRED: Macroeconomic Indicators

**Link:** https://fred.stlouisfed.org/

Series fetched:
- **CPIAUCSL** — Consumer Price Index for All Urban Consumers (monthly, seasonally adjusted)
- **DGS10** — 10-Year Treasury Constant Maturity Rate (daily)

In [7]:
import pandas_datareader.data as web

fred_series = {
    'CPI_US':       'CPIAUCSL',
    'Treasury_10Y': 'DGS10'
}

macro_data = {}

for name, series_id in fred_series.items():
    print(f'Fetching FRED series {series_id} ({name})...')
    try:
        df = web.DataReader(series_id, 'fred', START_DATE, END_DATE)
        df.index = pd.to_datetime(df.index)
        df.columns = [name]
        macro_data[name] = df
        fname = f'fred_{name.lower()}.csv'
        df.to_csv(fname)
        print(f'  {len(df)} rows | {df.index.min().date()} to {df.index.max().date()} | Saved {fname}')
    except Exception as e:
        print(f'  ERROR fetching {series_id}: {e}')

if macro_data:
    macro_combined = pd.concat(macro_data.values(), axis=1)
    macro_combined.to_csv('fred_macro.csv')
    print('\nSaved: fred_macro.csv')
    display(macro_combined.tail(5))

Fetching FRED series CPIAUCSL (CPI_US)...
  135 rows | 2015-01-01 to 2026-03-01 | Saved fred_cpi_us.csv
Fetching FRED series DGS10 (Treasury_10Y)...
  2947 rows | 2015-01-01 to 2026-04-17 | Saved fred_treasury_10y.csv

Saved: fred_macro.csv


,CPI_US,Treasury_10Y
DATE,,
2026-04-13,NaN,4.30
2026-04-14,NaN,4.26
2026-04-15,NaN,4.29
2026-04-16,NaN,4.32
2026-04-17,NaN,4.26


## Data Summary

In [8]:
print('=== Downloaded CSV Files ===')
for f in sorted(f for f in os.listdir('.') if f.endswith('.csv')):
    rows = len(pd.read_csv(f))
    print(f'  {f:<48}  {rows:>6} rows')

print('\n=== Apple Stock (head) ===')
if 'Apple' in price_data:
    display(price_data['Apple'].head(3))

print('\n=== Samsung Stock (head) ===')
if 'Samsung' in price_data:
    display(price_data['Samsung'].head(3))

print('\n=== Macro Data (head) ===')
if macro_data:
    display(macro_combined.head(3))

=== Downloaded CSV Files ===
  apple_balance_sheet.csv                               69 rows
  apple_cash_flow.csv                                   53 rows
  apple_income_stmt.csv                                 39 rows
  fred_cpi_us.csv                                      135 rows
  fred_macro.csv                                      2986 rows
  fred_treasury_10y.csv                               2947 rows
  samsung_balance_sheet.csv                             82 rows
  samsung_cash_flow.csv                                 59 rows
  samsung_income_stmt.csv                               47 rows
  stock_prices.csv                                    2929 rows

=== Apple Stock (head) ===


,Close,High,Low,Open,Volume
Date,,,,,
2015-01-02,24.214897,24.682230,23.776357,24.671155,212818400
2015-01-05,23.532717,24.064280,23.346671,23.984545,257142000
2015-01-06,23.534935,23.794071,23.173914,23.596950,263188400



=== Samsung Stock (head) ===


,Close,High,Low,Open,Volume
Date,,,,,
2015-01-02,20461.583984,20615.430480,20415.430036,20615.430480,8774950
2015-01-05,20507.734375,20553.888316,20200.041436,20553.888316,10139500
2015-01-06,19923.128906,20261.591328,19815.436318,20230.822017,15235500



=== Macro Data (head) ===


,CPI_US,Treasury_10Y
DATE,,
2015-01-01,234.747,NaN
2015-01-02,NaN,2.12
2015-01-05,NaN,2.04
